In [4]:
from kmc import KMCv2Runner, get_params_from_csv
from pathlib import Path

import numpy as np

In [5]:
out_dir = Path('../temp/output')
runs_iterable = range(32)
steps = 100_000
sigma_iterable = [x * 0.02 for x in range(8)]
dimensions = [
    [1000],
    [30,30],
    [10, 10, 10],
    [6, 6, 6, 6],
    [5, 5, 5, 5, 5]
]
lattice_vectors = [
    [1],
    [1,1],
    [1,1,1],
	[1,1,1,1],
	[1,1,1,1,1]
]

In [6]:
init_params, saddle_params = get_params_from_csv('../csv/1.csv', 'utf-8')
init_mean = init_params[0]
saddle_mean = saddle_params[0]
sets = {}
for i in range(len(dimensions)):
    sets[f'slopes_val_{i}'] = []
    sets[f'slopes_err_{i}'] = []

for sigma in sigma_iterable:
    this_dir = out_dir / f'{sigma:.2f}'
    init_params = [init_mean, sigma]
    saddle_params = [saddle_mean, sigma]
    init_params = [init_params for _ in range(len(dimensions))]
    saddle_params = [saddle_params for _ in range(len(dimensions))]
    runner = KMCv2Runner(this_dir, dimensions, lattice_vectors, init_params, saddle_params, 1000)
    print(f'Running for sigma = {sigma:.2f}')
    runner.run(steps, runs_iterable)
    slopes = runner.get_slopes()
    for i, slope_arr in enumerate(slopes):
        sets[f'slopes_val_{i}'].append(np.mean(slope_arr))
        sets[f'slopes_err_{i}'].append(np.std(slope_arr)/np.sqrt(len(slope_arr)))


Running for sigma = 0.00



Running KMCs: 100%|██████████| 160/160 [06:39<00:00,  2.50s/runs]


Running for sigma = 0.02



Running KMCs:  52%|█████▏    | 83/160 [03:52<03:13,  2.52s/runs]Process ForkPoolWorker-44:
Process ForkPoolWorker-45:
Process ForkPoolWorker-34:
Process ForkPoolWorker-33:
Process ForkPoolWorker-43:
Process ForkPoolWorker-41:
Process ForkPoolWorker-37:
Process ForkPoolWorker-38:
Process ForkPoolWorker-36:
Process ForkPoolWorker-42:
Process ForkPoolWorker-47:
Process ForkPoolWorker-46:
Traceback (most recent call last):


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
divisor = [sets[f'slopes_val_{i}'][0] for i in range(len(dimensions))]

divisor = [1 if x == 0 else x for x in divisor]
for i in range(len(dimensions)):
    plt.errorbar(sigma_iterable, np.array(sets[f'slopes_val_{i}'])/divisor[i],
                 yerr=np.array(sets[f'slopes_err_{i}'])/divisor[i],
                 label=f'{len(dimensions[i])}D')
plt.xlabel('Sigma')
plt.ylabel('Diffusivity')
plt.legend()
plt.show()